# Orfipy ORF Prediction

![Orfipy ORF Prediction](https://proto-bio.github.io/proto-assets/images/tool/orfipy/hero.png)

This notebook demonstrates `run_orfipy_prediction`, which identifies open reading frames in DNA sequences by scanning for start and stop codons across all reading frames. It covers single-sequence prediction, custom codon and strand configuration, batch processing, and result inspection.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("orfipy")
display_overview("orfipy")
display_docs_section("orfipy", "Background")

# ORFipy

[ORFipy](https://github.com/urmi-21/orfipy) is a fast Python implementation of [open reading frame](https://en.wikipedia.org/wiki/Open_reading_frame) (ORF) extraction developed by [Singh and Wurtele](https://github.com/urmi-21/orfipy) at the Iowa State University Bioinformatics and Computational Biology Program. It scans DNA sequences for ORFs across both strands by default, identifies every stretch bounded by a configurable set of start and stop codons, and reports the resulting ORFs together with their translated protein sequences. This toolkit exposes ORFipy through a single registered tool that accepts one or more DNA sequences and returns the ORFs per sequence with both nucleotide and amino-acid output.

ORFipy ([Singh and Wurtele, 2021](https://doi.org/10.1093/bioinformatics/btab090)) was developed as a fast and flexible replacement for older ORF-extraction tools that struggle with the scale of contemporary genomic and transcriptomic datasets. The published work emphasises customisable search criteria together with high throughput, and reports that ORFipy scales to whole-genome and de novo transcriptome inputs that exceed what earlier ORF finders can comfortably process. The reference implementation is written in Python and is distributed through PyPI and bioconda.

An [open reading frame](https://en.wikipedia.org/wiki/Open_reading_frame) is a continuous stretch of DNA bounded by an in-frame [start](https://en.wikipedia.org/wiki/Start_codon) and [stop codon](https://en.wikipedia.org/wiki/Stop_codon). ORF extraction is mechanistic rather than predictive. Every region that begins at a recognised start codon and continues in frame to the first downstream stop codon is reported, regardless of whether the resulting region encodes a biologically functional protein. This stands in contrast to gene-prediction tools such as Prodigal, which apply learned models to score whether each candidate ORF is likely to correspond to a real gene. ORFipy is appropriate when the goal is exhaustive enumeration of every candidate ORF for downstream filtering or annotation; a gene-prediction tool is appropriate when the goal is a curated set of likely coding genes.

## Available tools

In [2]:
display_available_tools("orfipy")

- **`run_orfipy_prediction()`** — ORF (Open Reading Frame) prediction using Orfipy

### `run_orfipy_prediction`

`run_orfipy_prediction` scans DNA sequences for open reading frames using Orfipy, a fast Cython-based ORF finder. For each input sequence it returns all ORFs that pass the configured start codon, stop codon, length, and strand filters. Each predicted ORF carries the amino acid translation, nucleotide sequence, 1-indexed coordinates, strand, reading frame, and GC content. Batch inputs are processed together and results are returned as a nested list keyed by input sequence, so downstream code can group or filter ORFs by their parent sequence.

In [3]:
from proto_tools import OrfipyInput, OrfipyConfig, OrfipyOutput, run_orfipy_prediction

In [4]:
# Display input docs
display_api_reference("orfipy", "input", "run_orfipy_prediction")

# A short DNA sequence with a known ORF (hemoglobin alpha fragment)
inputs = OrfipyInput(sequences="ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGGGCAAGTGA")

**Input** — `OrfipyInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA sequence(s) to analyze for open reading frames |

In [5]:
# Display config docs
display_api_reference("orfipy", "config", "run_orfipy_prediction")

# Forward strand only, ATG start codon only, minimum 30 nucleotides | see docs above for all fields
config = OrfipyConfig(
    start_codons=["ATG"],
    min_len=30,
    strand="f",
    include_stop=True,
)

**Config** — `OrfipyConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>threads</code> | <code>int</code> | <code>4</code> | CPU threads passed to orfipy --procs |
| <code>start_codons</code> | <code>list[Literal['ATG', 'GTG', 'TTG', 'CTG']]</code> | <code>['ATG', 'GTG', 'TTG']</code> | Start codons to recognize for ORF prediction |
| <code>stop_codons</code> | <code>list[Literal['TAA', 'TAG', 'TGA']]</code> | <code>['TAA', 'TAG', 'TGA']</code> | Stop codons to recognize for ORF prediction |
| <code>strand</code> | <code>Literal['f', 'r', 'b']</code> | <code>'b'</code> | Which strand(s) to scan: 'f' (forward), 'r' (reverse), or 'b' (both) |
| <code>min_len</code> | <code>int</code> | <code>0</code> | Min ORF length in nt; 0 keeps all ORFs |
| <code>max_len</code> | <code>int</code> | <code>10000</code> | Max ORF length in nt (caps payloads); raise to 1_000_000_000 for genome-scale |
| <code>include_stop</code> | <code>bool</code> | <code>True</code> | Include the stop codon in the reported ORF nucleotide sequence and length |
| <code>ignore_case</code> | <code>bool</code> | <code>False</code> | Treat lowercase (soft-masked) nucleotides as ORF-eligible |
| <code>partial_3</code> | <code>bool</code> | <code>False</code> | Report ORFs missing a stop codon at the sequence end |
| <code>partial_5</code> | <code>bool</code> | <code>False</code> | Report ORFs missing a start codon at the sequence start |
| <code>between_stops</code> | <code>bool</code> | <code>False</code> | Report ORFs spanning stop-to-stop (ignores start codons; implies partial_3 + partial_5) |
| <code>translation_table</code> | <code>Literal['standard', 'vertebrate_mitochondrial', 'yeast_mitochondrial', 'mold_protozoan_mitochondrial', 'invertebrate_mitochondrial', 'ciliate_nuclear', 'echinoderm_mitochondrial', 'euplotid_nuclear', 'bacterial', 'alternative_yeast_nuclear', 'ascidian_mitochondrial', 'alternative_flatworm_mitochondrial', 'chlorophycean_mitochondrial', 'trematode_mitochondrial', 'scenedesmus_mitochondrial', 'thraustochytrium_mitochondrial', 'rhabdopleuridae_mitochondrial', 'candidate_division_sr1', 'pachysolen_nuclear', 'karyorelict_nuclear', 'condylostoma_nuclear', 'mesodinium_nuclear', 'peritrich_nuclear'] &#124; None</code> | <code>None</code> | NCBI genetic code for translation (None = standard genetic code) |

In [6]:
# Run the prediction
result = run_orfipy_prediction(inputs, config)

Running run_orfipy_prediction [00:00]

In [7]:
# Display output docs
display_api_reference("orfipy", "output", "run_orfipy_prediction")

# Inspect predicted ORFs from the single input sequence
print(f"Found {result.num_orfs} ORFs")
for orf_list in result.predicted_orfs:
    for orf in orf_list:
        print(f"  {orf.id}: {orf.amino_acid_sequence} ({orf.amino_acid_length} aa, {orf.strand} strand, frame {orf.frame})")

# Demonstrate batch processing over multiple sequences
sequences = [
    "ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGGGCAAGTGA",  # human hemoglobin alpha CDS
    "ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGGGAAAACCCTGGCGTTACCCAACTTAATCGCCTTGCAGCACATCCCCCTTTCGCCAGCTGGCGTAATAGCGAAGAGGCCCGCACCGATCGCCCTTCCCAACAGTTGCGCAGCCTGAATGGCGAATGGCGCTTTGCCTGGTTTCCGGCACCAGAAGCGGTGCCG",  # E. coli lacZ N-terminal coding fragment
    "ATGAGTAAAGGAGAAGAACTTTTCACTGGAGTTGTCCCAATTCTTGTTGAATTAGATGGTGATGTTAATGGGCACAAATTTTCTGTCAGTGGAGAGGGTGAAGGTGATGCAACATACGGAAAACTTACCCTTAAATTTATTTGCACTACTGGAAAACTACCTGTTCCATGG",  # GFP N-terminal coding fragment
]
batch_inputs = OrfipyInput(
    sequences=sequences,
)
batch_config = OrfipyConfig(min_len=12)
batch_result = run_orfipy_prediction(batch_inputs, batch_config)

print(f"\nBatch: {batch_result.num_orfs} total ORFs across {len(sequences)} sequences")
print(f"ORFs per sequence: {batch_result.num_orfs_per_sequence}")

# Inspect an individual ORF
orf = batch_result.predicted_orfs[0][0]
print(f"\nFirst ORF details:")
print(f"  Parent ID:   {orf.parent_id}")
print(f"  Position:    {orf.nucleotide_start}-{orf.nucleotide_end}")
print(f"  AA length:   {orf.amino_acid_length}")
print(f"  GC content:  {orf.gc_content:.1f}%")
print(f"  Protein:     {orf.amino_acid_sequence}")

**Output** — `OrfipyOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>predicted_orfs</code> | <code>list[list[ORF]]</code> | <code>[]</code> | List of ORF results per input sequence |

**`ORF`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>parent_id</code> | <code>str</code> | required | Identifier of the parent/input sequence |
| <code>orf_id</code> | <code>str</code> | required | Unique ORF identifier within the parent sequence |
| <code>strand</code> | <code>Literal['+', '-']</code> | required | Strand direction ('+' or '-') |
| <code>frame</code> | <code>Literal[1, 2, 3]</code> | required | Reading frame (1, 2, or 3) |
| <code>amino_acid_sequence</code> | <code>str</code> | required | Translated protein sequence |
| <code>nucleotide_sequence</code> | <code>str</code> | required | DNA sequence of the ORF |
| <code>amino_acid_length</code> | <code>int</code> | required | Length of protein in amino acids |
| <code>nucleotide_length</code> | <code>int</code> | required | Length of ORF in nucleotides |
| <code>nucleotide_start</code> | <code>int</code> | required | Start position (1-indexed, inclusive) |
| <code>nucleotide_end</code> | <code>int</code> | required | End position (1-indexed, inclusive) |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Tool-specific metrics or metadata |

Found 1 ORFs
  seq_0_ORF.1: MVLSPADKTNVKAAWGK (17 aa, + strand, frame 1)


Running run_orfipy_prediction [00:00]


Batch: 9 total ORFs across 3 sequences
ORFs per sequence: [1, 3, 5]

First ORF details:
  Parent ID:   seq_0
  Position:    1-54
  AA length:   17
  GC content:  64.8%
  Protein:     MVLSPADKTNVKAAWGK


## Export Results

Predicted ORFs can be exported to CSV or JSON for tabular analysis, or to FASTA format (FAA for protein sequences, FNA for nucleotide sequences) for use in downstream bioinformatics pipelines.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

batch_result.export("orfipy_results", export_path=output_dir, file_format="csv")
print(f"Exported CSV to {output_dir / 'orfipy_results.csv'}")

batch_result.export("orfipy_results", export_path=output_dir, file_format="faa")
print(f"Exported protein FASTA to {output_dir / 'orfipy_results.faa'}")

Exported CSV to example_output/orfipy_results.csv
Exported protein FASTA to example_output/orfipy_results.faa
